# <center>EMPLOYEE LOAD</center>
```sql
SELECT
    e.BusinessEntityID, p.Title, p.FirstName, p.MiddleName, p.LastName, p.Suffix, e.JobTitle, 
    IIF(s.BusinessEntityID IS NULL, 0, 1) IsSalesPerson
    ,pp.PhoneNumber
    ,pnt.Name AS PhoneNumberType, ea.EmailAddress, p.EmailPromotion, a.AddressLine1, a.AddressLine2, a.City, 
    sp.Name AS StateProvinceName, a.PostalCode, cr.Name AS CountryRegionName, p.AdditionalContactInfo
FROM HumanResources.Employee e
	INNER JOIN Person.Person p
	ON p.BusinessEntityID = e.BusinessEntityID
    INNER JOIN Person.BusinessEntityAddress bea
    ON bea.BusinessEntityID = e.BusinessEntityID
    INNER JOIN Person.Address a
    ON a.AddressID = bea.AddressID
    INNER JOIN Person.StateProvince sp
    ON sp.StateProvinceID = a.StateProvinceID
    INNER JOIN Person.CountryRegion cr
    ON cr.CountryRegionCode = sp.CountryRegionCode
    LEFT OUTER JOIN Person.PersonPhone pp
    ON pp.BusinessEntityID = p.BusinessEntityID
    LEFT OUTER JOIN Person.PhoneNumberType pnt
    ON pp.PhoneNumberTypeID = pnt.PhoneNumberTypeID
    LEFT OUTER JOIN Person.EmailAddress ea
    ON p.BusinessEntityID = ea.BusinessEntityID
    LEFT OUTER JOIN Sales.SalesPerson s
    ON p.BusinessEntityID = s.BusinessEntityID;
```


In [ ]:
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_data_warehouse"
CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

staging_table_name = "staging.Integration.employee_Staging"
wh_table_name = "reporting.dimension.Employees"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog("local")

spark

```mermaid
flowchart
direction LR

Employee_BusinessEntityID -- INSERT --> staging_BusinessEntityID
Person_Title -- INSERT --> staging_Title
Person_FirstName -- INSERT --> staging_FirstName
Person_MiddleName -- INSERT --> staging_MiddleName
Person_LastName -- INSERT --> staging_LastName
Person_Suffix -- INSERT --> staging_Suffix
Employee_JobTitle -- INSERT --> staging_JobTitle
PersonPhone_PhoneNumber -- INSERT --> staging_PhoneNumber
PhoneNumberType_Name -- INSERT --> staging_PhoneNumberType
EmailAddress_EmailAddress -- INSERT --> staging_EmailAddress
Person_EmailPromotion -- INSERT --> staging_EmailPromotion
Address_AddressLine1 -- INSERT --> staging_AddressLine1
Address_AddressLine2 -- INSERT --> staging_AddressLine2
Address_City -- INSERT --> staging_City
StateProvince_Name -- INSERT --> staging_StateProvinceName
Address_PostalCode -- INSERT --> staging_PostalCode
CountryRegion_Name -- INSERT --> staging_CountryRegionName
Person_AdditionalContactInfo -- INSERT --> staging_AdditionalContactInfo
SalesPerson_isSalesPerson -- INSERT --> staging_IsSalesPerson

staging_BusinessEntityID ins1@-- UPSERT -->dimension_BusinessEntityID
staging_Title ins2@-- UPSERT -->dimension_Title
staging_FirstName ins3@-- UPSERT -->dimension_FirstName
staging_MiddleName ins4@-- UPSERT -->dimension_MiddleName
staging_LastName ins5@-- UPSERT -->dimension_LastName
staging_Suffix ins6@-- UPSERT -->dimension_Suffix
staging_JobTitle ins7@-- UPSERT -->dimension_JobTitle
staging_IsSalesPerson ins19@-- UPSERT -->dimension_IsSalesPerson
staging_PhoneNumber ins8@-- UPSERT -->dimension_PhoneNumber
staging_PhoneNumberType ins9@-- UPSERT -->dimension_PhoneNumberType
staging_EmailAddress ins10@-- UPSERT -->dimension_EmailAddress
staging_EmailPromotion in11@-- UPSERT -->dimension_EmailPromotion
staging_AddressLine1 ins12@-- UPSERT -->dimension_AddressLine1
staging_AddressLine2 ins13@-- UPSERT -->dimension_AddressLine2
staging_City ins14@-- UPSERT -->dimension_City
staging_StateProvinceName ins15@-- UPSERT -->dimension_StateProvinceName
staging_PostalCode ins16@-- UPSERT -->dimension_PostalCode
staging_CountryRegionName ins17@-- UPSERT -->dimension_CountryRegionName
staging_AdditionalContactInfo ins18@-- UPSERT -->dimension_AdditionalContactInfo

ins1@{animation: fast}
ins2@{animation: fast}
ins3@{animation: fast}
ins4@{animation: fast}
ins5@{animation: fast}
ins6@{animation: fast}
ins7@{animation: fast}
ins8@{animation: fast}
ins9@{animation: fast}
ins10@{animation: fast}
in11@{animation: fast}
ins12@{animation: fast}
ins13@{animation: fast}
ins14@{animation: fast}
ins15@{animation: fast}
ins16@{animation: fast}
ins17@{animation: fast}
ins18@{animation: fast}
ins19@{animation: fast}


Person_BusinessEntityID j1@ o-.Join.-o Employee_BusinessEntityID
BusinessEntityAddress_BusinessEntityID j2@ o-.Join.-o Employee_BusinessEntityID
Address_AddressID j3@ o-.Join.-o BusinessEntityAddress_AddressID
StateProvince_StateProvinceID j4@ o-.Join.-o Address_StateProvinceID
CountryRegion_CountryRegionCode j5@ o-.Join.-o StateProvince_CountryRegionCode
PersonPhone_BusinessEntityID j6@ o-.Join.-o Person_BusinessEntityID
PersonPhone_PhoneNumberTypeID j7@ o-.Join.-o PhoneNumberType_PhoneNumberTypeID
Person_BusinessEntityID j8@ o-.Join.-o EmailAddress_BusinessEntityID
Person_BusinessEntityID j9@ o-.Join.-o SalesPerson_BusinessEntityID
j1@{animation: slow}
j2@{animation: slow}
j3@{animation: slow}
j4@{animation: slow}
j5@{animation: slow}
j6@{animation: slow}
j7@{animation: slow}
j8@{animation: slow}
j9@{animation: slow}



subgraph source
    subgraph HumanResources.Employee
        direction TB
        Employee_JobTitle[JobTitle]
        Employee_BusinessEntityID[BusinessEntityID]
    end
    subgraph Person.Person
        direction TB
        Person_BusinessEntityID[BusinessEntityID] 
        Person_Title[Title]
        Person_FirstName[FirstName]
        Person_MiddleName[MiddleName]
        Person_LastName[LastName]
        Person_Suffix[Suffix]
        Person_EmailPromotion[EmailPromotion]
        Person_AdditionalContactInfo[AdditionalContactInfo]
    end
    subgraph Person.BusinessEntityAddress
        direction TB
        BusinessEntityAddress_BusinessEntityID[BusinessEntityID] 
        BusinessEntityAddress_AddressID[AddressID]
    end
    subgraph Person.PhoneNumberType
        direction TB
        PhoneNumberType_Name[Name]
        PhoneNumberType_PhoneNumberTypeID[PhoneNumberTypeID]
    end
    subgraph Person.Address
        direction TB
        Address_AddressID[AddressID]
        Address_StateProvinceID[StateProvinceID]
        Address_AddressLine1[AddressLine1]
        Address_AddressLine2[AddressLine2]
        Address_City[City]
        Address_PostalCode[PostalCode]
    end
    subgraph Person.StateProvince
        StateProvince_Name[Name]
        StateProvince_StateProvinceID[StateProvinceID]
        StateProvince_CountryRegionCode[CountryRegionCode]
    end
    subgraph Person.CountryRegion
        CountryRegion_Name[Name]
        CountryRegion_CountryRegionCode[CountryRegionCode]
    end
    subgraph Person.PersonPhone
        PersonPhone_PhoneNumber[PhoneNumber]
        PersonPhone_BusinessEntityID[BusinessEntityID]
        PersonPhone_PhoneNumberTypeID[PhoneNumberTypeID]
    end
    subgraph Person.PhoneNumberType
        PhoneNumberType_Name[PhoneNumberType]
        PhoneNumberType_PhoneNumberTypeID[PhoneNumberTypeID]
    end
    subgraph Person.EmailAddress
        direction TB
        EmailAddress_EmailAddress[EmailAddress]
        EmailAddress_BusinessEntityID[BusinessEntityID]
        EmailAddress_BusinessEntityID[BusinessEntityID]
    end
    subgraph Sales.SalesPerson
        direction TB
        SalesPerson_BusinessEntityID[BusinessEntityID]
        SalesPerson_isSalesPerson["IIF(s.BusinessEntityID IS NULL, 0, 1) IsSalesPerson"]
    end
end
subgraph destination
    subgraph staging_employees        
        staging_BusinessEntityID[BusinessEntityID]
        staging_Title[Title]
        staging_FirstName[FirstName]
        staging_MiddleName[MiddleName]
        staging_LastName[LastName]
        staging_Suffix[Suffix]
        staging_JobTitle[JobTitle]
        staging_IsSalesPerson[IsSalesPerson]
        staging_PhoneNumber[PhoneNumber]
        staging_PhoneNumberType[PhoneNumberType]
        staging_EmailAddress[EmailAddress]
        staging_EmailPromotion[EmailPromotion]
        staging_AddressLine1[AddressLine1]
        staging_AddressLine2[AddressLine2]
        staging_City[City]
        staging_StateProvinceName[StateProvinceName]
        staging_PostalCode[PostalCode]
        staging_CountryRegionName[CountryRegionName]
        staging_AdditionalContactInfo[AdditionalContactInfo]
    end
    subgraph dimension_employees
        dimension_BusinessEntityID[BusinessEntityID]
        dimension_Title[Title]
        dimension_FirstName[FirstName]
        dimension_MiddleName[MiddleName]
        dimension_LastName[LastName]
        dimension_Suffix[Suffix]
        dimension_JobTitle[JobTitle]
        dimension_IsSalesPerson[IsSalesPerson]
        dimension_PhoneNumber[PhoneNumber]
        dimension_PhoneNumberType[PhoneNumberType]
        dimension_EmailAddress[EmailAddress]
        dimension_EmailPromotion[EmailPromotion]
        dimension_AddressLine1[AddressLine1]
        dimension_AddressLine2[AddressLine2]
        dimension_City[City]
        dimension_StateProvinceName[StateProvinceName]
        dimension_PostalCode[PostalCode]
        dimension_CountryRegionName[CountryRegionName]
        dimension_AdditionalContactInfo[AdditionalContactInfo]
        dimension_recordHash[RecordHash]
        dimension_ValidFrom[ValidFrom]
        dimension_ValidTo[ValidTo]
        dimension_isActive[isActive]
    end
end
```

## <center> created staging table </center>

In [ ]:
df_dim_employee = spark.range(0)
df_dim_employee = df_dim_employee\
    .withColumn("EmployeeKey", sf.lit(None).cast(sdt.IntegerType()))\
    .withColumn("EmployeeID", sf.lit(None).cast(sdt.IntegerType()))\
    .withColumn("Title",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("FirstName",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("MiddleName",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("LastName",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("Suffix",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("JobTitle",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("isSalesPerson",sf.lit(None).cast(sdt.BooleanType()))\
    .withColumn("PhoneNumber",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("PhoneNumberType",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("EmailAddress",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("EmailPromotion",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("AddressLine1",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("AddressLine2",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("City",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("StateProvinceName",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("PostalCode",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("CountryRegionName",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("AdditionalContactInfo",sf.lit(None).cast(sdt.StringType()))\

df_dim_employee = df_dim_employee.drop("id")


df_dim_employee.writeTo(staging_table_name) \
    .using("iceberg") \
    .partitionedBy("CountryRegionName", "StateProvinceName", "City") \
    .tableProperty("format-version", "2") \
    .tableProperty("write.format.default", "parquet")\
    .createOrReplace()

spark.table(staging_table_name).show(10, False)


## <center> created destination table </center>

In [ ]:
df_dim_employee = spark.range(0)
df_dim_employee = df_dim_employee\
    .withColumn("EmployeeKey", sf.lit(None).cast(sdt.IntegerType()))\
    .withColumn("EmployeeID", sf.lit(None).cast(sdt.IntegerType()))\
    .withColumn("Title",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("FirstName",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("MiddleName",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("LastName",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("Suffix",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("JobTitle",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("isSalesPerson",sf.lit(None).cast(sdt.BooleanType()))\
    .withColumn("PhoneNumber",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("PhoneNumberType",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("EmailAddress",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("EmailPromotion",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("AddressLine1",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("AddressLine2",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("City",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("StateProvinceName",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("PostalCode",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("CountryRegionName",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("AdditionalContactInfo",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("recordHash",sf.lit(None).cast(sdt.StringType()))\
    .withColumn("validFrom",sf.lit(None).cast(sdt.DateType()))\
    .withColumn("validTill",sf.lit(None).cast(sdt.DateType()))\
    .withColumn("isActive",sf.lit(None).cast(sdt.BooleanType()))\

df_dim_employee = df_dim_employee.drop("id")


df_dim_employee.writeTo(wh_table_name) \
    .using("iceberg") \
    .partitionedBy("CountryRegionName", "StateProvinceName", "City") \
    .tableProperty("format-version", "2") \
    .tableProperty("write.format.default", "parquet")\
    .createOrReplace()

spark.table(wh_table_name).show(10, False)


In [ ]:
df_HumanResources_Employee = spark.table("HumanResources.Employee").alias("e")
df_Person_Person = spark.table("Person.Person").alias("p")
df_Person_BusinessEntityAddress = spark.table("Person.BusinessEntityAddress").alias("bea")
df_Person_Address = spark.table("Person.Address").alias("a")
df_Person_StateProvince = spark.table("Person.StateProvince").alias("sp")
df_Person_CountryRegion = spark.table("Person.CountryRegion").alias("cr")
df_Person_PersonPhone = spark.table("Person.PersonPhone").alias("pp")
df_Person_PhoneNumberType = spark.table("Person.PhoneNumberType").alias("pnt")
df_Person_EmailAddress = spark.table("Person.EmailAddress").alias("ea")
df_Sales_SalesPerson = spark.table("Sales.SalesPerson").alias("s")

In [ ]:
joined = (
    df_HumanResources_Employee
    .join(df_Person_Person, df_HumanResources_Employee.BusinessEntityID == df_Person_Person.BusinessEntityID, "inner")
    .join(df_Person_BusinessEntityAddress, df_Person_Person.BusinessEntityID == df_Person_BusinessEntityAddress.BusinessEntityID, "inner")
    .join(df_Person_Address, df_Person_BusinessEntityAddress.AddressID == df_Person_Address.AddressID, "inner")
    .join(df_Person_StateProvince, df_Person_Address.StateProvinceID == df_Person_StateProvince.StateProvinceID, "inner")
    .join(df_Person_CountryRegion, df_Person_StateProvince.CountryRegionCode == df_Person_CountryRegion.CountryRegionCode, "inner")
    .join(df_Person_PersonPhone, df_Person_Person.BusinessEntityID == df_Person_PersonPhone.BusinessEntityID, "left")
    .join(df_Person_PhoneNumberType, df_Person_PersonPhone.PhoneNumberTypeID == df_Person_PhoneNumberType.PhoneNumberTypeID, "left")
    .join(df_Person_EmailAddress, df_Person_Person.BusinessEntityID == df_Person_EmailAddress.BusinessEntityID, "left")
    .join(df_Sales_SalesPerson, df_Sales_SalesPerson.BusinessEntityID == df_Person_Person.BusinessEntityID, "left")
)\
    .select(
    sf.nullif(sf.col("e.BusinessEntityID"),sf.lit("NULL")).cast("int").alias("EmployeeID"),
    sf.nullif(sf.col("p.Title"),sf.lit("NULL")).alias("Title"),
    sf.nullif(sf.col("p.FirstName"),sf.lit("NULL")).alias("FirstName"),
    sf.nullif(sf.col("p.MiddleName"),sf.lit("NULL")).alias("MiddleName"),
    sf.nullif(sf.col("p.LastName"),sf.lit("NULL")).alias("LastName"),
    sf.nullif(sf.col("p.Suffix"),sf.lit("NULL")).alias("Suffix"),
    sf.nullif(sf.col("e.JobTitle"),sf.lit("NULL")).alias("JobTitle"),
    sf.when(sf.col("s.BusinessEntityID").isNull(), sf.lit(False)).otherwise(sf.lit(True)).alias("IsSalesPerson"),
    sf.nullif(sf.col("pp.PhoneNumber"),sf.lit("NULL")).alias("PhoneNumber"),
    sf.nullif(sf.col("pnt.Name"),sf.lit("NULL")).alias("PhoneNumberType"),
    sf.nullif(sf.col("ea.EmailAddress"),sf.lit("NULL")).alias("EmailAddress"),
    sf.nullif(sf.col("p.EmailPromotion"),sf.lit("NULL")).cast("int").alias("EmailPromotion"),
    sf.nullif(sf.col("a.AddressLine1"),sf.lit("NULL")).alias("AddressLine1"),
    sf.nullif(sf.col("a.AddressLine2"),sf.lit("NULL")).alias("AddressLine2"),
    sf.nullif(sf.col("a.City"),sf.lit("NULL")).alias("City"),
    sf.nullif(sf.col("sp.Name"),sf.lit("NULL")).alias("StateProvinceName"),
    sf.nullif(sf.col("a.PostalCode"),sf.lit("NULL")).alias("PostalCode"),
    sf.nullif(sf.col("cr.Name"),sf.lit("NULL")).alias("CountryRegionName"),
    sf.nullif(sf.col("p.AdditionalContactInfo"),sf.lit("NULL")).alias("AdditionalContactInfo"),
)\
    .filter(sf.col("EmployeeID")<= 55)

df_employeeStaging = joined.dropDuplicates(["EmployeeID"])

df_employeeStaging.show(10, False)
df_employeeStaging.printSchema()

# df_Update.write \
#     .format("iceberg") \
#     .mode("overwrite") \
#     .save(staging_table_name)

df_employeeStaging.writeTo(staging_table_name) \
    .partitionedBy("CountryRegionName", "StateProvinceName", "City") \
    .using("iceberg") \
    .createOrReplace()

In [ ]:
# # spark.sql("SHOW TBLPROPERTIES " + staging_table_name).show(truncate=False)
# spark.sql("DESCRIBE TABLE EXTENDED " + staging_table_name).show(30, truncate=False)
# spark.read.table(staging_table_name + ".partitions").show(truncate=False)

In [ ]:
spark.sql("SELECT COUNT(*) AS TotalRecords FROM " + staging_table_name + " where 1 = 1 and Title is NULL").show()

In [ ]:
df_source = spark.read.table(staging_table_name)
df_source = df_source\
    .withColumn("recordHash", sf.md5(sf.concat_ws("||",
        sf.col("EmployeeID").cast("string"),
        sf.col("Title"),
        sf.col("FirstName"),
        sf.col("MiddleName"),
        sf.col("LastName"),
        sf.col("Suffix"),
        sf.col("JobTitle"),
        sf.col("isSalesPerson").cast("string"),
        sf.col("PhoneNumber"),
        sf.col("PhoneNumberType"),
        sf.col("EmailAddress"),
        sf.col("EmailPromotion").cast("string"),
        sf.col("AddressLine1"),
        sf.col("AddressLine2"),
        sf.col("City"),
        sf.col("StateProvinceName"),
        sf.col("PostalCode"),
        sf.col("CountryRegionName"),
        sf.col("AdditionalContactInfo")
    )))\
    .withColumn("validFrom", sf.date_add(sf.lit(datetime.now()),-1))\
    .withColumn("validTill", sf.lit(None).cast(sdt.DateType()))\
    .withColumn("isActive", sf.lit(True))

df_source.show(10, False)

In [ ]:
df_destination = spark.read.table(wh_table_name)


raw_key = df_destination.agg(sf.max("EmployeeKey").alias("maxKey")).collect()[0]["maxKey"]
last_staging_key = raw_key if raw_key is not None else 0

seqNumber = sw.Window.orderBy(sf.lit(None))

df_source = df_source\
    .withColumn("EmployeeKey", sf.row_number().over(seqNumber) + last_staging_key)\


df_insert = df_source.alias("scrc")\
    .join(df_destination.alias("dest"), 
          (sf.col("scrc.EmployeeID") == sf.col("dest.EmployeeID")) & 
          (sf.col("scrc.recordHash") == sf.col("dest.recordHash"))&
          (sf.col("dest.isActive") == sf.lit(True)), 
          "left")\
    .filter(sf.col("dest.EmployeeID").isNull())\
    .select("scrc.*")



df_insert = df_insert\
    .select(
        "EmployeeKey",
        "EmployeeID",
        "Title",
        "FirstName",
        "MiddleName",
        "LastName",
        "Suffix",
        "JobTitle",
        "isSalesPerson",
        "PhoneNumber",
        "PhoneNumberType",
        "EmailAddress",
        "EmailPromotion",
        "AddressLine1",
        "AddressLine2",
        "City",
        "StateProvinceName",
        "PostalCode",
        "CountryRegionName",
        "AdditionalContactInfo",
        "recordHash",
        "validFrom",
        "validTill",
        "isActive"
    )

df_Update = df_insert.alias("scrc")\
    .join(df_destination.alias("dest"), 
          (sf.col("scrc.EmployeeID") == sf.col("dest.EmployeeID")) & 
          (sf.col("dest.isActive") == sf.lit(True)), 
          "inner")\
    .select("dest.*")
            


df_Update = df_Update\
    .withColumn("validTill", sf.date_add(sf.lit(datetime.now()),0))\
    .withColumn("isActive", sf.lit(False))

df_Update = df_Update\
    .select(
        "EmployeeKey",
        "EmployeeID",
        "Title",
        "FirstName",
        "MiddleName",
        "LastName",
        "Suffix",
        "JobTitle",
        "isSalesPerson",
        "PhoneNumber",
        "PhoneNumberType",
        "EmailAddress",
        "EmailPromotion",
        "AddressLine1",
        "AddressLine2",
        "City",
        "StateProvinceName",
        "PostalCode",
        "CountryRegionName",
        "AdditionalContactInfo",
        "recordHash",
        "validFrom",
        "validTill",
        "isActive"
    )


# df_destination.printSchema()
# df_insert.printSchema()
df_insert.show(10, False)
df_Update.show(10, False)




# df_Update.count()
# df_insert.count()


In [ ]:
df_Update.write \
    .format("iceberg") \
    .mode("overwrite") \
    .save(wh_table_name)

df_insert.write \
    .format("iceberg") \
    .mode("append") \
    .save(wh_table_name)


In [ ]:
df_dim_employee = spark.read.table(wh_table_name)
# df_dim_employee.printSchema()
df_dim_employee.show(10, False)

In [ ]:
spark.sql("Update " + staging_table_name + " SET Title = '' WHERE 1 = 1 and TITLE IS NULL and employeeID <= 55")
spark.table(staging_table_name)\
    .filter(sf.col("Title") == '')\
    .show(10, False)

In [ ]:
df_source = spark.read.table(staging_table_name)\
    .filter(sf.col("EmployeeID")<= 101)

df_source = df_source\
    .withColumn("recordHash", sf.md5(sf.concat_ws("||",
        sf.col("EmployeeID").cast("string"),
        sf.col("Title"),
        sf.col("FirstName"),
        sf.col("MiddleName"),
        sf.col("LastName"),
        sf.col("Suffix"),
        sf.col("JobTitle"),
        sf.col("isSalesPerson").cast("string"),
        sf.col("PhoneNumber"),
        sf.col("PhoneNumberType"),
        sf.col("EmailAddress"),
        sf.col("EmailPromotion").cast("string"),
        sf.col("AddressLine1"),
        sf.col("AddressLine2"),
        sf.col("City"),
        sf.col("StateProvinceName"),
        sf.col("PostalCode"),
        sf.col("CountryRegionName"),
        sf.col("AdditionalContactInfo")
    )))\
    .withColumn("validFrom", sf.date_add(sf.lit(datetime.now()),0))\
    .withColumn("validTill", sf.lit(None).cast(sdt.DateType()))\
    .withColumn("isActive", sf.lit(True))

df_source.show(10, False)

df_source.count()


In [ ]:
df_destination = spark.read.table(wh_table_name)


raw_key = df_destination.agg(sf.max("EmployeeKey").alias("maxKey")).collect()[0]["maxKey"]
last_staging_key = raw_key if raw_key is not None else 0

df_source = df_source\
    .withColumn("EmployeeKey", sf.row_number().over(seqNumber) + last_staging_key)\


df_insert = df_source.alias("scrc")\
    .join(df_destination.alias("dest"), 
          (sf.col("scrc.EmployeeID") == sf.col("dest.EmployeeID")) & 
          (sf.col("scrc.recordHash") == sf.col("dest.recordHash"))&
          (sf.col("dest.isActive") == sf.lit(True)), 
          "left")\
    .filter(sf.col("dest.EmployeeID").isNull())\
    .select("scrc.*")

seqNumber = sw.Window.orderBy(sf.lit(None))


# .cast(sdt.IntegerType())


df_insert = df_insert\
    .select(
        "EmployeeKey",
        "EmployeeID",
        "Title",
        "FirstName",
        "MiddleName",
        "LastName",
        "Suffix",
        "JobTitle",
        "isSalesPerson",
        "PhoneNumber",
        "PhoneNumberType",
        "EmailAddress",
        "EmailPromotion",
        "AddressLine1",
        "AddressLine2",
        "City",
        "StateProvinceName",
        "PostalCode",
        "CountryRegionName",
        "AdditionalContactInfo",
        "recordHash",
        "validFrom",
        "validTill",
        "isActive"
    )



df_Update = df_insert.alias("scrc")\
    .join(df_destination.alias("dest"), 
          (sf.col("scrc.EmployeeID") == sf.col("dest.EmployeeID")) & 
          (sf.col("dest.isActive") == sf.lit(True)), 
          "inner")\
    .select("dest.EmployeeKey", 
            "dest.EmployeeID",
            "dest.Title",
            "dest.FirstName",
            "dest.MiddleName",
            "dest.LastName",
            "dest.Suffix",
            "dest.JobTitle",
            "dest.isSalesPerson",
            "dest.PhoneNumber",
            "dest.PhoneNumberType",
            "dest.EmailAddress",
            "dest.EmailPromotion",
            "dest.AddressLine1",
            "dest.AddressLine2",
            "dest.City",
            "dest.StateProvinceName",
            "dest.PostalCode",
            "dest.CountryRegionName",
            "dest.AdditionalContactInfo",
            "dest.recordHash",
            "dest.validFrom",
            "dest.validTill",
            "dest.isActive",
             )
            


df_Update = df_Update\
    .withColumn("validTill", sf.date_add(sf.lit(datetime.now()),0))\
    .withColumn("isActive", sf.lit(False))

df_Update = df_Update\
    .select(
        "EmployeeKey",
        "EmployeeID",
        "Title",
        "FirstName",
        "MiddleName",
        "LastName",
        "Suffix",
        "JobTitle",
        "isSalesPerson",
        "PhoneNumber",
        "PhoneNumberType",
        "EmailAddress",
        "EmailPromotion",
        "AddressLine1",
        "AddressLine2",
        "City",
        "StateProvinceName",
        "PostalCode",
        "CountryRegionName",
        "AdditionalContactInfo",
        "recordHash",
        "validFrom",
        "validTill",
        "isActive"
    )


# df_destination.printSchema()

# df_insert.printSchema()

# df_insert.show(10, False)

# df_Update.show(10, False)




df_Update.count()
# df_insert.count()


In [ ]:
df_Update.filter(sf.col("EmployeeID") == 5).show(10, False)
df_insert.filter(sf.col("EmployeeID") == 5).show(10, False)

In [ ]:
# df_Update.write \
#     .format("iceberg") \
#     .mode("overwrite") \
#     .save(wh_table_name)

# df_insert.write \
#     .format("iceberg") \
#     .mode("append") \
#     .save(wh_table_name)


## Testing

In [ ]:
# df_dim_employee = spark.read.table(wh_table_name)
# df_dim_employee.filter(sf.col("EmployeeID") == 45).show(10, False)

In [ ]:
# spark.catalog.setCurrentCatalog("staging")
spark.stop()